TP NUM 3
Nom: DELICE CHAMUTU Gentil TP EDA
Promotion: BAC 2 Genie Minier
Cours: IA

#Question: a partir de donnees coffe_sales faites une analyse et traitement de vos donnees de maniere complete

ANALYSE ET EXPLORATION DE DONNEES

In [ ]:
import numpy as np  # calcul et algebre lineaire
import pandas as pd # gestion de la base de donnees 
import matplotlib.pyplot as plt # gestion des graphes  - visualisation
import seaborn as sns # toujours pour la visualisation

In [ ]:
# Style global des graphiques
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


CHARGEMENT DES DONNÉES

In [ ]:
file_path = r"C:/User/DELICE/Documents/Coffe_sales.csv" # Mets le fichier dans le même dossier que ce notebook
df = pd.read_csv(file_path)

In [ ]:
# Aperçu rapide
print("Dimensions :", df.shape)
display(df.head())
display(df.tail())

EXPLORATION INITIALE

In [ ]:
print("--- Informations générales ---")
df.info()
print("--- Noms des colonnes ---")
print(df.columns.tolist())
print("--- Valeurs manquantes ---")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])
print("--- Doublons ---")
print("Nombre de lignes dupliquées :", df.duplicated().sum())
print("--- Statistiques numériques ---")
display(df.describe(include=[np.number]).T)
print("--- Statistiques catégorielles ---")
display(df.describe(include=["object"]).T)

NETTOYAGE DES COLONNES

In [ ]:
# Nettoyage léger des noms de colonnes
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

In [ ]:
# Harmonisation si certaines colonnes existent sous des noms proches
rename_map = {
    "TimeofDay": "time_of_day",
    "Weekday": "weekday",
    "Monthname": "month_name",
    "DateTime": "date_time",
    "coffeename": "coffee_name",
    "cashtype": "cash_type",
    "hourofday": "hour_of_day",
    "Weekdaysort": "weekday_sort",
    "Monthsort": "month_sort",
    "money": "money"
}
df = df.rename(columns=rename_map)

CONVERSION DES TYPES

In [ ]:
# Convertir la date si elle existe
if "date_time" in df.columns:
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")


In [ ]:
# Convertir les colonnes numériques si besoin
for col in ["hour_of_day", "money", "weekday_sort", "month_sort"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
# Nettoyer les chaînes
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

CONTRÔLES QUALITÉ

In [ ]:
print("--- Après nettoyage ---")
df.info()

print("Dates invalides :", df["date_time"].isna().sum() if "date_time" in df.columns else "Pas de colonne date_time")


In [ ]:
# Suppression des lignes totalement inutilisables si nécessaire
df_clean = df.copy()
if "date_time" in df_clean.columns:
    df_clean = df_clean.dropna(subset=["date_time"])

CRÉATION DE VARIABLES UTILES

In [ ]:
if "date_time" in df_clean.columns:
    df_clean["date"] = df_clean["date_time"].dt.date
    df_clean["year"] = df_clean["date_time"].dt.year
    df_clean["month"] = df_clean["date_time"].dt.month
    df_clean["day"] = df_clean["date_time"].dt.day
    df_clean["day_name"] = df_clean["date_time"].dt.day_name()
    df_clean["hour"] = df_clean["date_time"].dt.hour

In [ ]:
# Normaliser le nom des boissons
if "coffee_name" in df_clean.columns:
    df_clean["coffee_name"] = df_clean["coffee_name"].str.lower()

ANALYSE DESCRIPTIVE

In [ ]:
print("--- Répartition des variables catégorielles ---")
for col in ["cash_type", "coffee_name", "time_of_day", "weekday", "month_name"]:
    if col in df_clean.columns:
        print(f"{col}")
        display(df_clean[col].value_counts(dropna=False).head(10))

print("--- Résumé des ventes ---")
if "money" in df_clean.columns:
    print("Total money :", round(df_clean["money"].sum(), 2))
    print("Moyenne :", round(df_clean["money"].mean(), 2))
    print("Médiane :", round(df_clean["money"].median(), 2))
    print("Min :", round(df_clean["money"].min(), 2))
    print("Max :", round(df_clean["money"].max(), 2))

TABLEAUX D'ANALYSE

In [ ]:
results = {}

if "coffee_name" in df_clean.columns and "money" in df_clean.columns:
    results["sales_by_coffee"] = (
        df_clean.groupby("coffee_name")["money"]
        .agg(["count", "sum", "mean"])
        .sort_values("sum", ascending=False)
    )

if "time_of_day" in df_clean.columns and "money" in df_clean.columns:
    results["sales_by_time"] = (
        df_clean.groupby("time_of_day")["money"]
        .agg(["count", "sum", "mean"])
        .sort_values("sum", ascending=False)
    )

if "weekday" in df_clean.columns and "money" in df_clean.columns:
    results["sales_by_weekday"] = (
        df_clean.groupby("weekday")["money"]
        .agg(["count", "sum", "mean"])
        .sort_values("sum", ascending=False)
    )

if "hour_of_day" in df_clean.columns and "money" in df_clean.columns:
    results["sales_by_hour"] = (
        df_clean.groupby("hour_of_day")["money"]
        .agg(["count", "sum", "mean"])
        .sort_index()
    )

for name, table in results.items():
    print(f"--- {name} ---")
    display(table.head(10))


VISUALISATION 1 - TOP BOISSONS

In [ ]:
if "coffee_name" in df_clean.columns and "money" in df_clean.columns:
    top_coffees = (
        df_clean.groupby("coffee_name")["money"]
        .sum()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(x=top_coffees.head(10).values, y=top_coffees.head(10).index, palette="viridis")
    plt.title("Top 10 des boissons par chiffre d'affaires")
    plt.xlabel("Chiffre d'affaires")
    plt.ylabel("Boisson")
    plt.tight_layout()
    plt.show()

VISUALISATION 2 - VENTES PAR HEURE

In [ ]:
if "hour_of_day" in df_clean.columns and "money" in df_clean.columns:
    hourly_sales = df_clean.groupby("hour_of_day")["money"].sum()

    plt.figure(figsize=(12, 6))
    sns.lineplot(x=hourly_sales.index, y=hourly_sales.values, marker="o")
    plt.title("Chiffre d'affaires par heure")
    plt.xlabel("Heure")
    plt.ylabel("Chiffre d'affaires")
    plt.xticks(range(int(df_clean["hour_of_day"].min()), int(df_clean["hour_of_day"].max()) + 1))
    plt.tight_layout()
    plt.show()


VISUALISATION 3 - VENTES PAR JOUR DE LA SEMAINE

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

if "day_name" in df_clean.columns and "money" in df_clean.columns:
    weekday_sales = df_clean.groupby("day_name")["money"].sum().reindex(weekday_order)

    plt.figure(figsize=(12, 6))
    sns.barplot(x=weekday_sales.index, y=weekday_sales.values, palette="magma")
    plt.title("Chiffre d'affaires par jour de la semaine")
    plt.xlabel("Jour")
    plt.ylabel("Chiffre d'affaires")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

VISUALISATION 4 - PÉRIODE DE LA JOURNÉE

In [ ]:
if "time_of_day" in df_clean.columns and "money" in df_clean.columns:
    period_order = ["Morning", "Afternoon", "Night"]
    period_sales = df_clean.groupby("time_of_day")["money"].sum().reindex(period_order)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=period_sales.index, y=period_sales.values, palette="coolwarm")
    plt.title("Chiffre d'affaires par période de la journée")
    plt.xlabel("Période")
    plt.ylabel("Chiffre d'affaires")
    plt.tight_layout()
    plt.show()

VISUALISATION 5 - CARTE DE CHALEUR

In [ ]:
if "day_name" in df_clean.columns and "hour_of_day" in df_clean.columns and "money" in df_clean.columns:
    heatmap_data = df_clean.pivot_table(
        index="day_name",
        columns="hour_of_day",
        values="money",
        aggfunc="sum"
    ).reindex(weekday_order)

    plt.figure(figsize=(14, 6))
    sns.heatmap(heatmap_data, cmap="YlOrRd", linewidths=0.3)
    plt.title("Heatmap des ventes par jour et par heure")
    plt.xlabel("Heure")
    plt.ylabel("Jour")
    plt.tight_layout()
    plt.show()

ANALYSE DES PRIX

In [ ]:
if "money" in df_clean.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df_clean["money"], bins=30, kde=True, color="teal")
    plt.title("Distribution des montants")
    plt.xlabel("Montant")
    plt.ylabel("Fréquence")
    plt.tight_layout()
    plt.show()

CORRÉLATIONS

In [ ]:
numeric_df = df_clean.select_dtypes(include=np.number)
if not numeric_df.empty:
    plt.figure(figsize=(10, 6))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="Blues", fmt=".2f")
    plt.title("Matrice de corrélation")
    plt.tight_layout()
    plt.show()


EXPORT DES RÉSULTATS

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

df_clean.to_csv(output_dir / "coffee_sales_cleaned.csv", index=False)

for name, table in results.items():
    table.to_csv(output_dir / f"{name}.csv")

print("Analyse terminée. Fichiers exportés dans le dossier output/.")